# Tennis Match Predictor

The goal of this project is to analyze different machine learning models for predicting the outcome of a tennis match.

## Contents

1. **Data loading** 
2. **Exploratory data analysis** 
3. **Data cleaning**
4. **Train/test split**
5. **Feature engineering**
6. **Training and evaluation** 

The implementation used in this notebook lives in the `src/` package.

# Setup & Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_data, train_test_split
from src.utils.utils import get_df_from_pipeline
from src.data.preprocess import (
    preprocessing_pipeline, 
    create_labels, 
    DropFeatureSelector
)
from src.features.features import feature_engineering_pipeline
from src.training.training import training_pipeline, fine_tune, feature_selection_pipeline
from src.evaluation.metrics import (
    cross_validation,
    evaluation_metrics,
    feature_importance,
    print_metrics
)
from src.evaluation.plots import (
    plot_confusion_matrix, 
    plot_roc_curve, 
    plot_feature_importance,
    plot_difference
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

# Data Loading

Loading professional (ATP) tennis single matches played between 2014 and 2025. Datasets are published by [tennis-data.co.uk](http://tennis-data.co.uk/) as one file per season.

`load_data()` concatenate them into a single DataFrame by merging raw files into a single chronologically sorted dataset based on match date

In [ ]:
raw_df : pd.DataFrame = load_data()
raw_df.head().transpose()

# Exploratory Data Analysis

Before proceeding with any data manipulation, we first examine the dataset to gain insights

## Columns

In [ ]:
print(raw_df.columns.tolist())

This dataset contains the following information about tennis matches:

- Tournament Information
    - ATP: Tournament number
    - Location: Venue of the tournament
    - Tournament: Name of the tournament
    - Date: Date of the match
    - Series: ATP tournament category
- Match Conditions
    - Court: Court type
    - Surface: Surface type
    - Round: Stage of the tournament
    - Best of: Maximum number of sets played in the match
- Players
    - Winner: Match winner
    - Loser: Match loser
    - WRank: ATP ranking of the winner at tournament start
    - LRank: ATP ranking of the loser at tournament start
    - WPts: ATP ranking points of the winner at tournament start
    - LPts: ATP ranking points of the loser at tournament start
- Match Score
    - W1, L1: Games won by winner/loser in the 1st set
    - W2, L2: Games won by winner/loser in the 2nd set
    - W3, L3: Games won by winner/loser in the 3rd set
    - W4, L4: Games won by winner/loser in the 4th set
    - W5, L5: Games won by winner/loser in the 5th set
    - Wsets: Total sets won by the match winner
    - Lsets: Total sets won by the match loser
    - Comment: Match outcome note
- Bets — odds for the winner (`*W`) and the loser (`*L`) of the match

## Values

As we can see from the output of `info()` function, we need to take care of missing values.

In [ ]:
raw_df.info()

In [ ]:
missing = raw_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

The code checks for rows where one betting-odds value (MaxW/MaxL or AvgW/AvgL) is present while its corresponding value is missing.

The result is empty, meaning there are no partially missing odds pairs in the dataset, therefore
we can safely use an imputer to fill missing odds values with a constant value.

In [ ]:
mask = (
    (raw_df["MaxW"].notna() & raw_df["MaxL"].isna()) |
    (raw_df["MaxW"].isna() & raw_df["MaxL"].notna()) |
    (raw_df["AvgW"].notna() & raw_df["AvgL"].isna()) |
    (raw_df["AvgW"].isna() & raw_df["AvgL"].notna()) 
)

print(raw_df[mask])

## Values Distributions

By looking at match statistics using the `describe()` function, we can gain some insights about the values in the dataset:
- Match Format:
    - Most matches are best of 3 sets, this also explains the high amount of null values in the `W4`, `L4`, `W5`, `L5` columns
- Rank/Points:
    - Matches in the dataset are mostly played by top 100 players, since they tipically play more matches in a tournament
    - The median of WRank and LRank indicates that winners are generally better ranked than losers. This statement is valid also for the points.
- Sets/Games:
    - Winning the first set is a strong predictor of winning the match, as the distribution of `W1` shows.
    - Over 50% of best of 3 matches don't go beyond 2 sets

In [ ]:
raw_df.describe().transpose()

Insights can be visualized with the following histograms

In [ ]:
raw_df.hist(figsize=(20, 15));

In [ ]:
# print(len(df[df["WRank"] > 1000]))
# print(len(df[df["LRank"] > 1000]))
# print(df[df["WRank"] > 1000][["Winner", "WRank", "Loser", "LRank"]])
# print(df[df["LRank"] > 1000][["Winner", "WRank", "Loser", "LRank"]])

Density of matches indicates that most matches are played by top ranked players

In [ ]:
raw_df.plot(kind="scatter", x="WRank", y="LRank", alpha=0.2);

### Winner - Loser Ranking plot

In [ ]:
plt.hist(raw_df["WRank"], bins=100, range=(1, 200), alpha=0.5, label='WRank')
plt.hist(raw_df["LRank"], bins=100, range=(1, 200), alpha=0.5, label='LRank')

plt.xlabel('Ranking')
plt.ylabel('Frequency')
plt.title('Winner - Loser Ranking')
plt.legend()

plt.show()

Higher ranking means more matches and more wins

In [ ]:
top_100 = raw_df[(raw_df["Date"] >= "2024-01-01") &(raw_df["WRank"] <= 100) & (raw_df["LRank"] <= 100)]
rank_diff = top_100["WRank"] - top_100["LRank"]

plt.scatter(top_100["WRank"], top_100["LRank"], alpha=0.2)
plt.plot([1, 100], [1, 100], color='red', linestyle='--')
plt.xlabel("Winner Rank (Top 100)")
plt.ylabel("Loser Rank (Top 100)")
plt.title("Winner vs Loser Rankings")

plt.show()

Here, we can see that the difference in ranking is more significant among higher-ranked players. Therefore, the same 20-position difference has a greater impact between players ranked 1st and 20th than between players ranked 80th and 100th.

## Values Range

### Comment

In [ ]:
raw_df["Comment"].value_counts(dropna=False)

The `Comment` column records how a match ended, these are the values that can appear:

- **Completed** — the match was played to its natural end, one player won the required number of sets on court. It's the only outcome whose score and statistics are fully reliable.
- **Retired** — the match started but a player withdrew while it was in progress (injury, illness). The winner is decided by the retirement, not by the score, so games and sets recorded are only the portion played before the withdrawal.
- **Walkover** — a player withdrew *before* the first point, so the match was never played: there is no score and the result carries no information about the two players.
- **Awarded** — the win was assigned by the officials after a mid-match default.
- **Disqualified** — same situation as *Awarded*, recorded under a different label by the data provider.
- **Sched** - the match was scheduled but never played, so there is no score and the result carries no information about the two players.
- **Rrtired** — a typo of *Retired* in the source file, normalized back to *Retired* during preprocessing

Therefore the matches which outcome was Walkover, Awarded, Sched or Disqualified do not provide any information about the match itself and since they are very few, we can drop them in the preprocessing step. 

### Surface

In [ ]:
raw_df["Surface"].value_counts(dropna=False)

### Round

In [ ]:
raw_df["Round"].value_counts(dropna=False)

### Court

In [ ]:
raw_df["Court"].value_counts(dropna=False)

### Series

In [ ]:
raw_df["Series"].value_counts(dropna=False)

### Tournament

In [ ]:
raw_df["Tournament"].value_counts(dropna=False)

## Data Cleaning

The raw data is recorded *from the outcome's point of view*: every row has a `Winner` and a `Loser`, and every statistic of the match comes as a winner/loser pair (`WRank`/`LRank`, `WPts`/`LPts`, `MaxW`/`MaxL`, ...). Therefore the answer is encoded in the column layout itself, and a model trained on it as-is would simply learn "the first player always wins".

To solve the problem, in the function `create_lables()` each match is re-encoded around two general players, `player1` and `player2`, assigned with a fair coin flip. Every winner/loser column pair is swapped consistently based on the coin flip, and the target becomes

$$\texttt{winner} = \begin{cases} 1 & \text{if player1 won the match} \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
cleaned_df = raw_df.copy()
create_labels(cleaned_df)
cleaned_df["winner"].value_counts(dropna=False)

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(cleaned_df)
years = X_train["Date"].dt.year.max() - X_train["Date"].dt.year.min() + 1

`train_test_split` (in `src/data/loader.py`) is performed with a 80/20 ratio based on the match date for avoiding data leakage

## Preprocessing

In [ ]:
cleaned_X_train, cleaned_y_train = X_train.copy(), y_train.copy()
prep_pipeline = preprocessing_pipeline(X=cleaned_X_train, y=cleaned_y_train)
cleaned_X_train = get_df_from_pipeline(prep_pipeline, cleaned_X_train)

`preprocessing_pipeline` builds the scikit-learn `Pipeline` that turns the raw DataFrame into a cleaned one.

### Row filtering (before the pipeline)

Some rows are removed in place from `X`/`y` *before* the pipeline is assembled if they satisfy one of these conditions:

- **Comment** (`Comment == "Awarded" or Comment == "Walkover" or Comment == "Disqualified" or Comment == "Sched"`), matches that were partially played and whose "result" says nothing about the two players, as we said above these are very few and can be dropped without losing much information.
- **matches with no set count** (`player1_sets`/`player2_sets` missing), rows too incomplete to update player history reliably.

### The pipeline steps

**1. `drop_cols` — `DropFeatureSelector`.** Drops bet columns with high rate of missing values

**2. `preprocessing` — `ColumnTransformer`.** Imputation and encoding, applied per column block:

| block | columns | what it does |
| --- | --- | --- |
| `zero_imputer` | games + sets | fills with `0`, a missing game score means the set was simply never played (most matches are best of 3) |
| `bet_imputer` | the odds columns | fills with `2.0`, equal odd for both players |
| `rank_imputer` | `player1_Rank`, `player2_Rank` | fills with the worst rank observed in the training frame |
| `point_imputer` | `player1_Pts`, `player2_Pts` | fills with the smallest points observed | 
| `categorical_encoder` | `Tournament` | one-hot encoding |
| `round_encoder` | `Round` | ordinal encoding along the real tournament with order: Round Robin → 1st Round → ... → The Final |
| `series_encoder` | `Series` | ordinal encoding along the tournament importance with order: ATP250 → ATP500 → Masters 1000 → Masters Cup → Grand Slam |

**3. `best_of_imputer` — `BestOfImputer`.** `Best of` is missing on part of the dataset, but it is recoverable verifying the sets played

As we can see from the output of `info()` function all columns contain no missing values

In [ ]:
cleaned_X_train.info(verbose=True, show_counts=True)

In [ ]:
print(cleaned_X_train.isnull().any(axis=1).sum())

The `Comment` column records have been successfully corrected

In [ ]:
X_train["Comment"].value_counts()

In [ ]:
cleaned_X_train["Comment"].value_counts()

## Feature Engineering

In [ ]:
feat_X_train, feat_y_train = cleaned_X_train.copy(), cleaned_y_train.copy()
feat_pipeline = feature_engineering_pipeline()
feat_X_train = get_df_from_pipeline(feat_pipeline, feat_X_train)

`feature_engineering_pipeline` builds a `Pipeline` with a single step, `FeatureEngineringTransformer`, that replaces the per-match statistics with features describing the two players as they arrive at the match.

The transformer iterate through the matches in chronological order and keeps a `Player` object per athlete (`src/utils/player.py`) and build the following features for each match:

| feature | definition |
| --- | --- |
| `rank_diff` | ATP rank difference at match time |
| `h2h_diff` | head-to-head difference, matches won by `player1` against `player2` minus the opposite |
| `h2h_weighted_diff` | weighted head-to-head difference giving more importance to recent meetings |
| `h2h_court_diff` | head-to-head difference specifically on the same court type |
| `h2h_surface_diff` | head-to-head difference specifically on the same surface |
| `sets_h2h_diff` | same difference counted in sets won in their previous meetings |
| `win_rate_diff` | career win rate observed so far (`wins / matches`, `0.5` for a player with no history) |
| `win_rate_bestof3_diff` | win rate difference in best-of-3 format matches |
| `win_rate_bestof5_diff` | win rate difference in best-of-5 format matches |
| `last_k_matches_win_rate_diff` | win rate over the last `k = 20` matches (`0.5` if no history) defining the recent form |
| `last_k_sets_win_rate_diff` | Win rate over the last `k = 20` matches' sets, indicating set-level form |
| `last_k_matches_rank_variation_diff` | difference in rank variation over the last k matches, indicating trend in performance |
| `win_streak_diff` | consecutive wins currently held by each player |
| `lose_streak_diff` | consecutive losses currently held by each player |
| `surface_win_rate_diff` | win rate on the surface of the match (`0.5` if never played on it). |
| `court_win_rate_diff` | win rate on the court type of the match (`0.5` if never played on it). |
| `fatigue_diff` | difference of a fatigue score built from the previous match. |
| `number_recent_matches_diff` | difference in the number of matches played in a recent time window. |
| `max_bet_diff` | best odds offered across bookmakers |
| `avg_bet_diff` | average odds across bookmakers |
| `weighted_ranking_diff` | Weighted ranking difference accounting for ranking points |


### Columns carried over

Three columns are carried over from the preprocessing step:

- `round` and `series`, the ordinal encodings of the tournament stage and of the tournament importance, they describe the *context* of the match rather than the players.
- the `Tournament_*` one-hot columns

Everything else, the per-set games, the sets, the points, the raw odds and the player names, is consumed to build the features and the player histories and does not reach the model.

In [ ]:
feat_X_train.info(verbose=True, show_counts=True)

## Training & Evaluation

Definition of the pipeline built so far

In [ ]:
preprocessor = Pipeline(
    steps=[
        ("preprocessing", prep_pipeline),
        ("feature_engineering", feat_pipeline),
    ]
)

## Random Forest

Firstly the we add the Random Forest Classifier as a step for the pipeline, then the `fine_tune()` method is used to find the best hyperparameters using a `RandomizedSearchCV`, finally the pipeline is fitted with the training data. 

In [ ]:
random_forest_train_pipeline = training_pipeline(model=RandomForestClassifier(random_state=42))

pipeline = Pipeline(
    steps = preprocessor.steps + random_forest_train_pipeline.steps
)

hyperparams = {
        'training__n_estimators': [100, 200, 300],
        'training__max_depth': [None, 10, 20],
        'training__min_samples_split': [2, 5, 10],
        'training__min_samples_leaf': [1, 2, 4]
    }

fine_tuned_rf_pipeline = fine_tune(
    pipeline=pipeline,
    tuning_methods=RandomizedSearchCV,
    years=years,
    hyperparams=hyperparams,
    scoring="accuracy",
    verbose=1,
    random_state=42
    )

fine_tuned_rf_pipeline.fit(X_train, y_train)

### Cross-Validation

In [ ]:
# cross_val_scores = cross_validation(
#         pipeline=fine_tuned_rf_pipeline,
#         X=X_train,
#         y=y_train,
#         years=years,
#         verbose=0,
#         scoring="accuracy"
#     )

# print(pd.Series(cross_val_scores).describe())

### Feature Selection

In [ ]:
importances = feature_importance(
    pipeline=fine_tuned_rf_pipeline,
    X=X_train
)

importances

In [ ]:
plot_feature_importance(
    feature_names=importances.index.tolist(),
    feature_importances=importances["importance"],
    top_n=30
)

In [ ]:
pipeline = Pipeline(
        steps= preprocessor.steps + [
            ("drop_tournaments", DropFeatureSelector(features_to_drop=["Tournament_"], is_grouped=True)),
            ] + random_forest_train_pipeline.steps
    )

hyperparams = {
        'training__n_estimators': [100, 200, 300],
        'training__max_depth': [None, 10, 20],
        'training__min_samples_split': [2, 5, 10],
        'training__min_samples_leaf': [1, 2, 4]
    }

fine_tuned_pipeline : RandomizedSearchCV | GridSearchCV = fine_tune(
    pipeline=pipeline,
    tuning_methods=RandomizedSearchCV,
    years=years,
    hyperparams=hyperparams,
    scoring="accuracy",
    verbose=1,
    random_state=42
    )

fine_tuned_pipeline.fit(X_train, y_train)

In [ ]:
# cross_val_scores = cross_validation(
#         pipeline=fine_tuned_pipeline,
#         X=X_train,
#         y=y_train,
#         years=years,
#         verbose=0,
#         scoring="accuracy"
#     )

# print(pd.Series(cross_val_scores).describe())

In [ ]:
rcevf_pipeline = feature_selection_pipeline(
    model=RandomForestClassifier(random_state=42),
    years=years
)

pipeline = Pipeline(
    steps= preprocessor.steps + [
            ("drop_tournaments", DropFeatureSelector(features_to_drop=["Tournament_"], is_grouped=True)),
            ("fs", rcevf_pipeline),
            ]
    )

X_train_fs = pipeline.fit_transform(X_train, y_train)

In [ ]:
hyperparams = {
        'training__n_estimators': [100, 200, 300],
        'training__max_depth': [None, 10, 20],
        'training__min_samples_split': [2, 5, 10],
        'training__min_samples_leaf': [1, 2, 4]
    }

search = fine_tune(
    pipeline=random_forest_train_pipeline,
    tuning_methods=RandomizedSearchCV,
    years=years,
    hyperparams=hyperparams,
    scoring="accuracy",
    verbose=1,
    random_state=42
    )

search.fit(X_train_fs, y_train)

In [ ]:
cross_val_scores = cross_validation(
        pipeline=search,
        X=X_train_fs,
        y=y_train,
        years=years,
        verbose=0,
        scoring="accuracy"
    )

print(pd.Series(cross_val_scores).describe())

### Evaluation

In [ ]:
y_pred = fine_tuned_pipeline.predict(X_test)

We are interested in accuracy most.

In [ ]:
metrics : dict = evaluation_metrics(
    model=fine_tuned_pipeline,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred
)

print_metrics(metrics)

In [ ]:
plot_confusion_matrix(cm=metrics["confusion_matrix"])

In [ ]:
plot_roc_curve(
    model=fine_tuned_pipeline,
    X_test=X_test,
    y_test=y_test
)

In [ ]:
wrong = np.asarray(y_pred) != np.asarray(y_test)
errors = X_test.loc[wrong].copy()
errors["y_true"] = np.asarray(y_test)[wrong]
errors["y_pred"] = np.asarray(y_pred)[wrong]

errors["y_true"].describe().transpose()

In [ ]:
proba = fitted_pipeline.predict_proba(X_test)[:, 1]
errors["proba_pos"] = proba[wrong]

# false positives vs false negatives
fp = errors[(errors.y_true == 0) & (errors.y_pred == 1)] # player_2 actually won
fn = errors[(errors.y_true == 1) & (errors.y_pred == 0)] # player_1 actually won

fp.describe().transpose()

In [ ]:
plt.hist(fp["player1_Rank"], bins=100, range=(1, 200), alpha=0.5, label='Loser Rank')
plt.hist(fp["player2_Rank"], bins=100, range=(1, 200), alpha=0.5, label='Winner Rank')

plt.xlabel('Ranking')
plt.ylabel('Frequency')
plt.title('Winner - Loser Ranking')
plt.legend()

plt.show()

In [ ]:
plot_difference(
    dataframe=fp,
    first_col="player1_Rank",  
    second_col="player2_Rank",
)

## Logistic Regression

In [ ]:
pipeline = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("standard_scaler", StandardScaler()),
            ("train", training_pipeline(model=LogisticRegression(random_state=42)))
        ]
    )

# hyperparams = [
#     {   # lbfgs / newton-cg: L2 or none only
#         "clf__solver": ["lbfgs"],
#         "clf__penalty": ["l2", None],
#         "clf__C": np.logspace(-4, 4, 9),
#         "clf__class_weight": [None, "balanced"],
#     },
#     {   # liblinear: L1 or L2, no None
#         "clf__solver": ["liblinear"],
#         "clf__penalty": ["l1", "l2"],
#         "clf__C": np.logspace(-4, 4, 9),
#         "clf__class_weight": [None, "balanced"],
#     },
#     {   # saga: everything, incl. elasticnet
#         "clf__solver": ["saga"],
#         "clf__penalty": ["elasticnet"],
#         "clf__C": np.logspace(-4, 4, 9),
#         "clf__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
#         "clf__class_weight": [None, "balanced"],
#     },
# ]

# fitted_pipeline : RandomizedSearchCV | GridSearchCV = fine_tune(
#     pipeline=pipeline,
#     tuning_methods=RandomizedSearchCV,
#     years=years,
#     hyperparams=hyperparams,
#     scoring="accuracy",
#     verbose=1,
#     random_state=42
#     )

pipeline.fit(X_train, y_train)

cross_val_scores = cross_validation(
        pipeline=pipeline,
        X=X_train,
        y=y_train,
        years=years,
        verbose=0,
        scoring="accuracy"
    )

print(pd.Series(cross_val_scores).describe())

## XGBoost

In [ ]:
pipeline = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("train", training_pipeline(model=xgb.XGBClassifier(random_state=42)))
        ]
    )

hyperparams = {
        'train__model__n_estimators': [100, 300, 500],
        'train__model__max_depth': [3, 6, 10],
        'train__model__learning_rate': [0.01, 0.05, 0.1],
        'train__model__subsample': [0.6, 0.8, 1.0],
        'train__model__colsample_bytree': [0.6, 0.8, 1.0],
        'train__model__min_child_weight': [1, 3, 5],
        'train__model__gamma': [0, 0.1, 0.3],
        'train__model__reg_lambda': [1, 5, 10]
    }

fitted_pipeline : RandomizedSearchCV | GridSearchCV = fine_tune(
    pipeline=pipeline,
    tuning_methods=RandomizedSearchCV,
    years=years,
    hyperparams=hyperparams,
    scoring="accuracy",
    random_state=42
)

fitted_pipeline.fit(X_train, y_train)

In [ ]:
cross_val_scores = cross_validation(
        pipeline=fitted_pipeline,
        X=X_train,
        y=y_train,
        years=years,
        verbose=0,
        scoring="accuracy"
    )

print(pd.Series(cross_val_scores).describe())